In [189]:
import glob
import os
import yaml
import geopandas as gpd
import pandas as pd

In [190]:
SEVERITY_LABEL = {0: "none", 1: "minor", 2: "serious", 3: "fatal"}


def load_config(config_path="../configs/data_sources.yaml"):
    with open(config_path, "r") as f:
        return yaml.safe_load(f)


def _severity(row) -> int:
    if row["ผู้เสียชีวิต"] >= 1:
        return 3
    if row["ผู้บาดเจ็บสาหัส"] >= 1:
        return 2
    if row["ผู้บาดเจ็บเล็กน้อย"] >= 1:
        return 1
    return 0


def _time_bin(h: int) -> str:
    if   6  <= h < 9:  return "morning_peak"
    elif 9  <= h < 16: return "daytime"
    elif 16 <= h < 20: return "evening_peak"
    elif 20 <= h < 24: return "night"
    else:              return "late_night"


In [191]:
config        = load_config()
bbox          = config["bangkok_bbox"]
acc_cfg       = config["accidents"]
raw_dir       = acc_cfg["raw_dir"]
out_gpkg      = acc_cfg["clean_gpkg"]
out_parquet   = acc_cfg["clean_parquet"]
projected_crs = acc_cfg["projected_crs"]
valid_weather = set(acc_cfg["valid_weather"])


In [192]:
raw_files = sorted(glob.glob(f"../{raw_dir}/accidents_*.csv"))

In [193]:
raw_files

['../data/raw/MOT_accident_data/accidents_2019.csv',
 '../data/raw/MOT_accident_data/accidents_2020.csv',
 '../data/raw/MOT_accident_data/accidents_2021.csv',
 '../data/raw/MOT_accident_data/accidents_2022.csv',
 '../data/raw/MOT_accident_data/accidents_2023.csv',
 '../data/raw/MOT_accident_data/accidents_2024.csv',
 '../data/raw/MOT_accident_data/accidents_2025.csv']

In [194]:
frames = [pd.read_csv(f, encoding="utf-8-sig", low_memory=False) for f in raw_files]
df = pd.concat(frames, ignore_index=True)
print(f"  Total rows (all Thailand): {len(df):,}")

  Total rows (all Thailand): 133,193


In [195]:
frames[0].head()

,_id,ปีที่เกิดเหตุ,วันที่เกิดเหตุ,เวลา,วันที่รายงาน,เวลาที่รายงาน,ACC_CODE,หน่วยงาน,สายทางหน่วยงาน,รหัสสายทาง,...,รถบรรทุก6ล้อ,รถบรรทุกไม่เกิน10ล้อ,รถบรรทุกมากกว่า10ล้อ,รถอีแต๋น,รถอื่นๆ,คนเดินเท้า,ผู้เสียชีวิต,ผู้บาดเจ็บสาหัส,ผู้บาดเจ็บเล็กน้อย,รวมจำนวนผู้บาดเจ็บ
0,1,2019,2019-01-01T00:00:00,0:00,2019-01-02T00:00:00,6:11,571905,กรมทางหลวงชนบท,ทางหลวงชนบท,ลบ.2029,...,0,0,0,0,0,0,0,0,2,2
1,2,2019,2019-01-01T00:00:00,0:03,2020-02-20T00:00:00,13:48,3790870,กรมทางหลวง,ทางหลวง,24,...,0,0,0,0,0,0,0,0,2,2
2,3,2019,2019-01-01T00:00:00,0:05,2019-01-01T00:00:00,10:35,599075,กรมทางหลวง,ทางหลวง,3168,...,0,0,0,0,0,0,1,0,0,0
3,4,2019,2019-01-01T00:00:00,0:20,2019-01-02T00:00:00,5:12,571924,กรมทางหลวงชนบท,ทางหลวงชนบท,ชม.4016,...,0,0,0,0,0,0,0,0,1,1
4,5,2019,2019-01-01T00:00:00,0:25,2019-01-04T00:00:00,9:42,599523,กรมทางหลวง,ทางหลวง,225,...,0,0,0,0,0,0,0,0,0,0


In [196]:
frames[1].head()

,_id,ปีที่เกิดเหตุ,วันที่เกิดเหตุ,เวลา,วันที่รายงาน,เวลาที่รายงาน,ACC_CODE,หน่วยงาน,สายทางหน่วยงาน,รหัสสายทาง,...,รถบรรทุก6ล้อ,รถบรรทุกไม่เกิน10ล้อ,รถบรรทุกมากกว่า10ล้อ,รถอีแต๋น,รถอื่นๆ,คนเดินเท้า,ผู้เสียชีวิต,ผู้บาดเจ็บสาหัส,ผู้บาดเจ็บเล็กน้อย,รวมจำนวนผู้บาดเจ็บ
0,1,2020,2020-01-01T00:00:00,0:05,2020-01-01T00:00:00,8:07,3785916,กรมทางหลวง,ทางหลวง,4225,...,0,0,0,0,0,1,0,1,0,1
1,2,2020,2020-01-01T00:00:00,0:05,2020-01-02T00:00:00,0:36,3786314,กรมทางหลวง,ทางหลวง,3473,...,0,0,0,0,0,0,0,1,0,1
2,3,2020,2020-01-01T00:00:00,0:15,2020-01-02T00:00:00,6:06,575019,กรมทางหลวงชนบท,ทางหลวงชนบท,นภ.3038,...,0,0,0,0,0,0,1,0,0,0
3,4,2020,2020-01-01T00:00:00,0:15,2021-01-20T00:00:00,7:46,3814981,กรมทางหลวง,ทางหลวง,302,...,0,0,0,0,0,0,0,0,1,1
4,5,2020,2020-01-01T00:00:00,0:20,2020-02-03T00:00:00,13:23,3789099,กรมทางหลวง,ทางหลวง,2196,...,0,0,0,0,0,0,0,1,0,1


In [197]:
frames[2].head()

,_id,ปีที่เกิดเหตุ,วันที่เกิดเหตุ,เวลา,วันที่รายงาน,เวลาที่รายงาน,ACC_CODE,หน่วยงาน,สายทางหน่วยงาน,รหัสสายทาง,...,รถบรรทุก6ล้อ,รถบรรทุกไม่เกิน10ล้อ,รถบรรทุกมากกว่า10ล้อ,รถอีแต๋น,รถอื่นๆ,คนเดินเท้า,ผู้เสียชีวิต,ผู้บาดเจ็บสาหัส,ผู้บาดเจ็บเล็กน้อย,รวมจำนวนผู้บาดเจ็บ
0,1,2021,2021-01-01T00:00:00,0:00,2021-06-02T00:00:00,9:04,4172437,กรมทางหลวง,ทางหลวง,2044,...,0,0,0,0,0,0,0,0,2,2
1,2,2021,2021-01-01T00:00:00,0:02,2021-01-02T00:00:00,6:31,2918300,กรมทางหลวงชนบท,ทางหลวงชนบท,สน.3054,...,0,0,0,0,0,0,0,0,1,1
2,3,2021,2021-01-01T00:00:00,0:05,2021-01-02T00:00:00,6:28,2918203,กรมทางหลวงชนบท,ทางหลวงชนบท,นพ.3015,...,0,0,0,0,0,0,0,0,1,1
3,4,2021,2021-01-01T00:00:00,0:05,2021-01-01T00:00:00,10:16,3812049,กรมทางหลวง,ทางหลวง,4169,...,0,0,0,0,0,0,0,0,5,5
4,5,2021,2021-01-01T00:00:00,0:05,2021-01-04T00:00:00,9:40,3812534,กรมทางหลวง,ทางหลวง,1143,...,0,0,0,0,0,0,1,2,0,2


In [198]:
frames[3].head()

,_id,ปีที่เกิดเหตุ,วันที่เกิดเหตุ,เวลา,วันที่รายงาน,เวลาที่รายงาน,ACC_CODE,หน่วยงาน,สายทางหน่วยงาน,รหัสสายทาง,...,รถบรรทุก6ล้อ,รถบรรทุกไม่เกิน10ล้อ,รถบรรทุกมากกว่า10ล้อ,รถอีแต๋น,รถอื่นๆ,คนเดินเท้า,ผู้เสียชีวิต,ผู้บาดเจ็บสาหัส,ผู้บาดเจ็บเล็กน้อย,รวมจำนวนผู้บาดเจ็บ
0,1,2022,2022-01-01T00:00:00,0:01,2022-01-02T00:00:00,11:45,6566872,กรมทางหลวงชนบท,ทางหลวงชนบท,ชน.5016,...,0,0,0,0,0,0,0,1,0,1
1,2,2022,2022-01-01T00:00:00,0:01,2022-01-02T00:00:00,11:44,6566880,กรมทางหลวงชนบท,ทางหลวงชนบท,มค.4012,...,0,0,0,0,0,0,0,0,1,1
2,3,2022,2022-01-01T00:00:00,0:03,2022-02-09T00:00:00,8:41,5706553,กรมทางหลวง,ทางหลวง,4,...,0,0,0,0,0,0,1,0,0,0
3,4,2022,2022-01-01T00:00:00,0:05,2022-01-02T00:00:00,6:21,5485750,กรมทางหลวง,ทางหลวง,4030,...,0,0,0,0,0,0,0,0,1,1
4,5,2022,2022-01-01T00:00:00,0:05,2022-01-24T00:00:00,9:59,5624452,กรมทางหลวง,ทางหลวง,216,...,0,0,0,0,0,0,0,0,2,2


In [199]:
for i, frame in enumerate(frames):
  if i < 4:
    frame['วันที่เกิดเหตุ'] = pd.to_datetime(frame['วันที่เกิดเหตุ'])
    frame['เวลา'] = pd.to_timedelta(frame['เวลา'] + ':00')

    frame['วันที่และเวลาเกิดเหตุ'] = frame['วันที่เกิดเหตุ'] + frame['เวลา']
    frame = frame.drop(columns=["วันที่เกิดเหตุ", "เวลา", "วันที่รายงาน", "เวลาที่รายงาน"], errors="ignore")

In [200]:
frames[0].head()

,_id,ปีที่เกิดเหตุ,วันที่เกิดเหตุ,เวลา,วันที่รายงาน,เวลาที่รายงาน,ACC_CODE,หน่วยงาน,สายทางหน่วยงาน,รหัสสายทาง,...,รถบรรทุกไม่เกิน10ล้อ,รถบรรทุกมากกว่า10ล้อ,รถอีแต๋น,รถอื่นๆ,คนเดินเท้า,ผู้เสียชีวิต,ผู้บาดเจ็บสาหัส,ผู้บาดเจ็บเล็กน้อย,รวมจำนวนผู้บาดเจ็บ,วันที่และเวลาเกิดเหตุ
0,1,2019,2019-01-01,0 days 00:00:00,2019-01-02T00:00:00,6:11,571905,กรมทางหลวงชนบท,ทางหลวงชนบท,ลบ.2029,...,0,0,0,0,0,0,0,2,2,2019-01-01 00:00:00
1,2,2019,2019-01-01,0 days 00:03:00,2020-02-20T00:00:00,13:48,3790870,กรมทางหลวง,ทางหลวง,24,...,0,0,0,0,0,0,0,2,2,2019-01-01 00:03:00
2,3,2019,2019-01-01,0 days 00:05:00,2019-01-01T00:00:00,10:35,599075,กรมทางหลวง,ทางหลวง,3168,...,0,0,0,0,0,1,0,0,0,2019-01-01 00:05:00
3,4,2019,2019-01-01,0 days 00:20:00,2019-01-02T00:00:00,5:12,571924,กรมทางหลวงชนบท,ทางหลวงชนบท,ชม.4016,...,0,0,0,0,0,0,0,1,1,2019-01-01 00:20:00
4,5,2019,2019-01-01,0 days 00:25:00,2019-01-04T00:00:00,9:42,599523,กรมทางหลวง,ทางหลวง,225,...,0,0,0,0,0,0,0,0,0,2019-01-01 00:25:00


In [201]:
frames[4]

,_id,ปีที่เกิดเหตุ,วันที่เกิดเหตุ,เวลา,วันที่รายงาน,เวลาที่รายงาน,ACC_CODE,หน่วยงาน,สายทางหน่วยงาน,รหัสสายทาง,...,รถบรรทุก6ล้อ,รถบรรทุกไม่เกิน10ล้อ,รถบรรทุกมากกว่า10ล้อ,รถอีแต๋น,รถอื่นๆ,คนเดินเท้า,ผู้เสียชีวิต,ผู้บาดเจ็บสาหัส,ผู้บาดเจ็บเล็กน้อย,รวมจำนวนผู้บาดเจ็บ
0,1,2023,1/1/2023,0:01,1/3/2023,12:05,7542138,กรมทางหลวงชนบท,ทางหลวงชนบท,รน.4038,...,0,0,0,0,0,0,0,1,0,1
1,2,2023,1/1/2023,0:05,1/3/2023,12:04,7542110,กรมทางหลวงชนบท,ทางหลวงชนบท,สส.5007,...,0,0,0,0,0,0,0,0,1,1
2,3,2023,1/1/2023,0:05,11/30/2023,14:02,8548373,กรมทางหลวง,ทางหลวง,4009,...,0,0,0,0,0,0,0,0,2,2
3,4,2023,1/1/2023,0:07,3/30/2023,14:33,7615338,กรมทางหลวง,ทางหลวง,2005,...,0,0,0,0,0,0,1,0,0,0
4,5,2023,1/1/2023,0:10,1/2/2023,5:42,7536059,กรมทางหลวง,ทางหลวง,1112,...,0,0,0,0,1,1,1,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
23836,23837,2023,12/31/2023,21:50,1/1/2024,12:27,9065312,กรมทางหลวงชนบท,ทางหลวงชนบท,ตง.5039,...,0,0,0,0,0,1,0,0,1,1
23837,23838,2023,12/31/2023,22:10,1/1/2024,11:48,9065337,กรมทางหลวงชนบท,ทางหลวงชนบท,ตร.1043,...,0,0,0,0,0,0,0,0,1,1
23838,23839,2023,12/31/2023,22:35,1/1/2024,5:52,9065317,กรมทางหลวงชนบท,ทางหลวงชนบท,ปข.2057,...,0,0,0,0,0,0,0,0,1,1
23839,23840,2023,12/31/2023,23:15,1/1/2024,4:56,9065304,กรมทางหลวงชนบท,ทางหลวงชนบท,ปจ.3016,...,0,0,0,0,0,0,0,0,1,1


In [202]:
frames[4]['วันที่เกิดเหตุ'] = pd.to_datetime(
    frames[4]['วันที่เกิดเหตุ'], format='%m/%d/%Y'
)

frames[4]['เวลา'] = pd.to_timedelta(frames[4]['เวลา'] + ':00')
frames[4]['วันที่และเวลาเกิดเหตุ'] = frames[4]['วันที่เกิดเหตุ'] + frames[4]['เวลา']

frames[4] = frames[4].drop(columns=["วันที่เกิดเหตุ", "เวลา", "วันที่รายงาน", "เวลาที่รายงาน"], errors="ignore")

In [203]:
frames[4].head()

,_id,ปีที่เกิดเหตุ,ACC_CODE,หน่วยงาน,สายทางหน่วยงาน,รหัสสายทาง,สายทาง,KM,จังหวัด,รถคันที่1,...,รถบรรทุกไม่เกิน10ล้อ,รถบรรทุกมากกว่า10ล้อ,รถอีแต๋น,รถอื่นๆ,คนเดินเท้า,ผู้เสียชีวิต,ผู้บาดเจ็บสาหัส,ผู้บาดเจ็บเล็กน้อย,รวมจำนวนผู้บาดเจ็บ,วันที่และเวลาเกิดเหตุ
0,1,2023,7542138,กรมทางหลวงชนบท,ทางหลวงชนบท,รน.4038,แยกทางหลวงหมายเลข 4005 (กม.ที่ 1+000) - บ้านทุ...,0.07,ระนอง,รถจักรยานยนต์,...,0,0,0,0,0,0,1,0,1,2023-01-01 00:01:00
1,2,2023,7542110,กรมทางหลวงชนบท,ทางหลวงชนบท,สส.5007,แยกทางหลวงชนบท สค.2055 (กม.ที่ 0+040) - บ้านคล...,0.70,สมุทรสงคราม,รถจักรยานยนต์,...,0,0,0,0,0,0,0,1,1,2023-01-01 00:05:00
2,3,2023,8548373,กรมทางหลวง,ทางหลวง,4009,เวียงสระ - บางสวรรค์,95.00,สุราษฎร์ธานี,รถปิคอัพบรรทุก 4 ล้อ,...,0,0,0,0,0,0,0,2,2,2023-01-01 00:05:00
3,4,2023,7615338,กรมทางหลวง,ทางหลวง,2005,หล่มเก่า - วังบาล,5.50,เพชรบูรณ์,รถจักรยานยนต์,...,0,0,0,0,0,1,0,0,0,2023-01-01 00:07:00
4,5,2023,7536059,กรมทางหลวง,ทางหลวง,1112,สลกบาตร - บ่อถ้ำ,3.00,กำแพงเพชร,อื่นๆ,...,0,0,0,1,1,1,0,0,0,2023-01-01 00:10:00


In [204]:
frames[5].head()

,_id,ปีที่เกิดเหตุ,วันที่เกิดเหตุ,เวลา,วันที่รายงาน,เวลาที่รายงาน,ACC_CODE,หน่วยงาน,สายทางหน่วยงาน,รหัสสายทาง,...,รถบรรทุก6ล้อ,รถบรรทุกไม่เกิน10ล้อ,รถบรรทุกมากกว่า10ล้อ,รถอีแต๋น,รถอื่นๆ,คนเดินเท้า,ผู้เสียชีวิต,ผู้บาดเจ็บสาหัส,ผู้บาดเจ็บเล็กน้อย,รวมจำนวนผู้บาดเจ็บ
0,1,2024,45292,0.008333,45461,0.599306,9701543,กรมทางหลวง,ทางหลวง,4164,...,0,0,0,0,0,0,0,0,1,1
1,2,2024,45292,0.020833,45292,0.509028,8901889,กรมทางหลวง,ทางหลวง,106,...,0,0,0,0,0,0,0,0,1,1
2,3,2024,45292,0.020833,45293,0.179167,8902334,กรมทางหลวง,ทางหลวง,1143,...,0,0,0,0,0,0,0,0,1,1
3,4,2024,45292,0.020833,45292,0.486111,8902375,กรมทางหลวง,ทางหลวง,3390,...,0,0,0,0,0,2,0,1,1,2
4,5,2024,45292,0.020833,45293,0.215278,8902450,กรมทางหลวง,ทางหลวง,4021,...,0,0,0,0,0,0,0,0,1,1


In [205]:
frames[5]['วันที่เกิดเหตุ'] = pd.to_datetime(frames[5]['วันที่เกิดเหตุ'], origin='1899-12-30', unit='D')
frames[5]['เวลา'] = pd.to_timedelta(frames[5]['เวลา'], unit='D')
frames[5]['วันที่และเวลาเกิดเหตุ'] = frames[5]['วันที่เกิดเหตุ'] + frames[5]['เวลา']
frames[5]['วันที่และเวลาเกิดเหตุ'] = frames[5]['วันที่และเวลาเกิดเหตุ'].dt.round('S')

frames[5] = frames[5].drop(columns=["วันที่เกิดเหตุ", "เวลา", "วันที่รายงาน", "เวลาที่รายงาน"], errors="ignore")

/var/folders/8c/ttrqjvgn62x_53dxr6sk5pg40000gn/T/ipykernel_65108/3201009877.py:4: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  frames[5]['วันที่และเวลาเกิดเหตุ'] = frames[5]['วันที่และเวลาเกิดเหตุ'].dt.round('S')


In [206]:
frames[5]

,_id,ปีที่เกิดเหตุ,ACC_CODE,หน่วยงาน,สายทางหน่วยงาน,รหัสสายทาง,สายทาง,KM,จังหวัด,รถคันที่1,...,รถบรรทุกไม่เกิน10ล้อ,รถบรรทุกมากกว่า10ล้อ,รถอีแต๋น,รถอื่นๆ,คนเดินเท้า,ผู้เสียชีวิต,ผู้บาดเจ็บสาหัส,ผู้บาดเจ็บเล็กน้อย,รวมจำนวนผู้บาดเจ็บ,วันที่และเวลาเกิดเหตุ
0,1,2024,9701543,กรมทางหลวง,ทางหลวง,4164,NaN,17.200,พัทลุง,รถจักรยานยนต์,...,0,0,0,0,0,0,0,1,1,2024-01-01 00:12:00
1,2,2024,8901889,กรมทางหลวง,ทางหลวง,106,ลี้ - ม่วงโตน,161.891,ลำพูน,รถจักรยานยนต์,...,0,0,0,0,0,0,0,1,1,2024-01-01 00:30:00
2,3,2024,8902334,กรมทางหลวง,ทางหลวง,1143,น้ำคลาด - ปางหมิ่น,52.000,พิษณุโลก,รถจักรยานยนต์,...,0,0,0,0,0,0,0,1,1,2024-01-01 00:30:00
3,4,2024,8902375,กรมทางหลวง,ทางหลวง,3390,หนองรี - บ่อยาง,13.700,กาญจนบุรี,รถจักรยานยนต์,...,0,0,0,0,2,0,1,1,2,2024-01-01 00:30:00
4,5,2024,8902450,กรมทางหลวง,ทางหลวง,4021,เมืองภูเก็ต - ห้าแยกฉลอง,4.800,ภูเก็ต,รถจักรยานยนต์,...,0,0,0,0,0,0,0,1,1,2024-01-01 00:30:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
23947,23948,2024,9965800,กรมทางหลวง,ทางหลวง,3201,เนินสันติ - แยกยายรวย,8.000,ชุมพร,รถปิคอัพบรรทุก 4 ล้อ,...,0,0,0,0,0,0,0,1,1,2024-12-31 22:50:00
23948,23949,2024,9965753,กรมทางหลวง,ทางหลวง,3032,บุ้งกี๋ - ท่าศาล,23.230,NaN,รถปิคอัพบรรทุก 4 ล้อ,...,0,0,0,0,0,0,1,0,1,2024-12-31 23:10:00
23949,23950,2024,9966849,กรมทางหลวง,ทางหลวง,2244,มอดินแดง - ศรีเทพน้อย,7.050,NaN,รถจักรยานยนต์,...,0,0,0,0,0,0,1,2,3,2024-12-31 23:15:00
23950,23951,2024,9965770,กรมทางหลวง,ทางหลวง,3086,NaN,16.900,NaN,รถจักรยานยนต์,...,0,0,0,0,0,0,1,0,1,2024-12-31 23:30:00


In [ ]:
frames[6].head()

,_id,ปีที่เกิดเหตุ,วันที่เกิดเหตุ,เวลา,วันที่รายงาน,เวลาที่รายงาน,ACC_CODE,หน่วยงาน,สายทางหน่วยงาน,รหัสสายทาง,...,รถบรรทุก6ล้อ,รถบรรทุกไม่เกิน10ล้อ,รถบรรทุกมากกว่า10ล้อ,รถอีแต๋น,รถอื่นๆ,คนเดินเท้า,ผู้เสียชีวิต,ผู้บาดเจ็บสาหัส,ผู้บาดเจ็บเล็กน้อย,รวมจำนวนผู้บาดเจ็บ
0,1,2025,45658,0:00,45658,20:31,9966403,กรมทางหลวง,ทางหลวง,347,...,0,0,0,0,0,0,0,0,1,1
1,2,2025,45658,0:01,45658,19:59,9966330,กรมทางหลวง,ทางหลวง,331,...,0,0,0,0,0,0,0,0,0,0
2,3,2025,45658,0:01,45659,5:01,9995820,กรมทางหลวงชนบท,ทางหลวงชนบท,สต.3007,...,0,0,0,0,0,0,0,0,1,1
3,4,2025,45658,0:03,45659,12:24,9995832,กรมทางหลวงชนบท,ทางหลวงชนบท,ส.012,...,0,0,0,0,0,0,0,0,1,1
4,5,2025,45658,0:03,45765,9:04,10041560,กรมทางหลวง,ทางหลวง,3220,...,0,0,0,0,0,0,0,4,0,4


In [209]:
frames[6]['วันที่เกิดเหตุ'] = pd.to_datetime(frames[6]['วันที่เกิดเหตุ'], origin='1899-12-30', unit='D')
frames[6]['เวลา'] = pd.to_timedelta(frames[6]['เวลา'] + ':00')

frames[6]['วันที่และเวลาเกิดเหตุ'] = frames[6]['วันที่เกิดเหตุ'] + frames[6]['เวลา']
frames[6] = frames[6].drop(columns=["วันที่เกิดเหตุ", "เวลา", "วันที่รายงาน", "เวลาที่รายงาน"], errors="ignore")

In [210]:
frames[6].head()

,_id,ปีที่เกิดเหตุ,ACC_CODE,หน่วยงาน,สายทางหน่วยงาน,รหัสสายทาง,สายทาง,KM,จังหวัด,รถคันที่1,...,รถบรรทุกไม่เกิน10ล้อ,รถบรรทุกมากกว่า10ล้อ,รถอีแต๋น,รถอื่นๆ,คนเดินเท้า,ผู้เสียชีวิต,ผู้บาดเจ็บสาหัส,ผู้บาดเจ็บเล็กน้อย,รวมจำนวนผู้บาดเจ็บ,วันที่และเวลาเกิดเหตุ
0,1,2025,9966403,กรมทางหลวง,ทางหลวง,347,NaN,50.832,อยุธยา,รถปิคอัพบรรทุก 4 ล้อ,...,0,0,0,0,0,0,0,1,1,2025-01-01 00:00:00
1,2,2025,9966330,กรมทางหลวง,ทางหลวง,331,เนินโมก - แปลงยาว,88.200,ชลบุรี,รถยนต์นั่งส่วนบุคคล/รถยนต์นั่งสาธารณะ,...,0,0,0,0,0,0,0,0,0,2025-01-01 00:01:00
2,3,2025,9995820,กรมทางหลวงชนบท,ทางหลวงชนบท,สต.3007,แยกทางหลวงหมายเลข 404 (กม.ที่ 93+100) - บ้านสะ...,13.500,สตูล,รถจักรยานยนต์,...,0,0,0,0,0,0,0,1,1,2025-01-01 00:01:00
3,4,2025,9995832,กรมทางหลวงชนบท,ทางหลวงชนบท,ส.012,จุดเชื่อมต่อสะพานภูมิพลที่ 1 เข้าถนนวงแหวนอุตส...,0.666,สมุทรปราการ,รถจักรยานยนต์,...,0,0,0,0,0,0,0,1,1,2025-01-01 00:03:00
4,5,2025,10041560,กรมทางหลวง,ทางหลวง,3220,แยกสะแกกรัง - เขาพะแวง,7.515,อุทัยธานี,รถยนต์นั่งส่วนบุคคล/รถยนต์นั่งสาธารณะ,...,0,0,0,0,0,0,4,0,4,2025-01-01 00:03:00
